# ⚡ Interview Questions: SQL Performance & Optimization
## From Slow Queries to Production-Scale Performance

### 🎯 Why Performance Optimization Separates Senior from Principal Engineers

**Writing queries that work is junior-level. Writing queries that scale to billions of rows is senior-level.** Here's why optimization matters:

1. **Cost Impact** - Slow queries cost real money (compute, storage, time)
2. **User Experience** - 10-second query vs 100ms query = retention vs churn
3. **Production Readiness** - Queries tested on 1000 rows fail on 1B rows
4. **Interview Differentiator** - 90% of candidates can write SQL, 10% can optimize it
5. **Career Growth** - Performance expertise = promotion to senior/principal roles

### 💡 What Separates Good from Great Engineers

| Good Engineer | Great Engineer |
|---------------|----------------|
| Writes correct queries | Writes correct AND fast queries |
| "Query works on sample data" | "Query scales to production data" |
| Uses indexes blindly | Understands index selectivity and cardinality |
| Knows EXPLAIN exists | Reads and interprets execution plans |
| Adds indexes when slow | Knows when NOT to add indexes |
| Fixes performance after complaints | Prevents performance issues before deployment |

---

### 📊 Interview Question Coverage (20 Questions)

This module covers **7 critical optimization domains**:

| Topic | Questions | Why It Matters |
|-------|-----------|----------------|
| **Query Execution Plans** | 3 | EXPLAIN, understanding optimizer decisions |
| **Indexing Strategies** | 4 | When to index, composite indexes, covering indexes |
| **JOIN Optimization** | 3 | Join order, broadcast vs shuffle, statistics |
| **Partitioning & Clustering** | 3 | Data layout for query performance |
| **Query Rewriting** | 3 | Transforming slow queries to fast ones |
| **Statistics & Cardinality** | 2 | How optimizer makes decisions |
| **Anti-Patterns** | 2 | Common mistakes that kill performance |

---

### 🎓 How to Master This Module

1. **Think like the optimizer** - Understand how queries are executed
2. **Measure everything** - Profile before and after optimization
3. **Know your data** - Cardinality, distribution, skew affect plans
4. **Test at scale** - 1000 rows ≠ 1B rows
5. **Understand trade-offs** - Faster queries may use more memory/storage

### 🏆 Interview Success Tips

✅ **Always ask about data size** - "How many rows? Growth rate?"
✅ **Explain the WHY** - "This index helps because..."
✅ **Mention trade-offs** - "Faster reads but slower writes"
✅ **Profile first** - "Let me check EXPLAIN before optimizing"
✅ **Think distributed** - "At scale, this becomes a shuffle operation"

⚠️ **Red flags that fail interviews:**
- Can't read an EXPLAIN plan
- Suggests "add more indexes" without analysis
- Doesn't know about index selectivity
- Never heard of query statistics or cardinality
- Can't explain why a query is slow
- Doesn't consider data skew in distributed systems

---

**Ready to master SQL performance? Let's dive in!** 🚀

## 🔍 Section 1: Query Execution Plans (3 Questions)

Execution plans show HOW the database executes your query. Reading them is essential for optimization.

### ❓ Question 1: Understanding EXPLAIN Plans

**Fundamental Interview Question:**
> "Explain what an EXPLAIN plan is and how to read it. What are the key metrics you look for when diagnosing slow queries? Show how you'd optimize a slow JOIN query."

### ✅ Answer 1: Reading and Interpreting EXPLAIN Plans

#### **What is an EXPLAIN Plan?**

An EXPLAIN plan shows:
- **How** the database will execute your query
- **What operations** it will perform (scan, join, filter, aggregate)
- **In what order** operations happen
- **Cost estimates** (rows processed, memory, I/O)

#### **Basic EXPLAIN Syntax:**

```sql
-- Standard SQL
EXPLAIN SELECT * FROM orders WHERE customer_id = 123;

-- Databricks/Spark SQL
EXPLAIN FORMATTED SELECT ...;
EXPLAIN EXTENDED SELECT ...;
EXPLAIN COST SELECT ...;
```

#### **Key Metrics to Look For:**

**1. Table Scans (⚠️ Often Slow)**
```
Table Scan: orders
  Rows: 10,000,000
  Filters: customer_id = 123
```
**Problem:** Reading entire table to find 1 customer
**Fix:** Add index on customer_id

**2. Index Usage (✅ Usually Fast)**
```
Index Seek: idx_customer_id
  Rows: 5
  Key Lookup: Get remaining columns
```
**Good:** Only reading relevant rows

**3. Join Methods**
```
Nested Loop Join      -- Good for small datasets
Hash Join             -- Good for medium datasets
Merge Join            -- Good when inputs are sorted
Broadcast Join        -- Good for small dimension tables (Spark)
Shuffle Hash Join     -- Expensive in distributed systems (Spark)
```

**4. Row Estimates**
```
Estimated Rows: 1,000,000
Actual Rows: 50
```
**Problem:** Huge mismatch = optimizer chose wrong plan
**Fix:** Update statistics

**5. Sort Operations (💰 Expensive)**
```
Sort: ORDER BY created_date
  Rows: 10,000,000
  Memory: 500MB
```
**Problem:** Sorting millions of rows in memory
**Fix:** Index on created_date, or reduce rows before sorting

#### **Sample EXPLAIN Plan Analysis:**

**Slow Query:**
```sql
SELECT c.name, COUNT(o.order_id)
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_date >= '2024-01-01'
GROUP BY c.name;
```

**EXPLAIN Output (Simplified):**
```
1. Table Scan: customers (1M rows)
2. Table Scan: orders (10M rows)
3. Hash Join on customer_id
   - Build: customers (1M rows)
   - Probe: orders (10M rows)
4. Filter: order_date >= '2024-01-01' (after join!)
5. Aggregate: GROUP BY name
6. Result: 50K rows
```

**Problems:**
1. **Table scans** on both tables
2. **Filter applied AFTER join** (processing 10M rows unnecessarily)
3. **No indexes used**

**Optimized Query:**
```sql
SELECT c.name, COUNT(o.order_id)
FROM customers c
JOIN (
  SELECT customer_id, order_id
  FROM orders
  WHERE order_date >= '2024-01-01'  -- Filter BEFORE join
) o ON c.customer_id = o.customer_id
GROUP BY c.name;
```

**With Indexes:**
```sql
CREATE INDEX idx_orders_date_customer ON orders(order_date, customer_id);
CREATE INDEX idx_customers_id ON customers(customer_id);
```

**New EXPLAIN Output:**
```
1. Index Seek: orders.idx_orders_date_customer
   - Rows: 500K (filtered by date)
2. Index Seek: customers.idx_customers_id
   - Rows: 1M
3. Hash Join on customer_id
   - Build: customers (1M rows)
   - Probe: orders (500K rows)  ← 20x fewer rows!
4. Aggregate: GROUP BY name
5. Result: 50K rows
```

**Improvement:** 10M → 500K rows joined = ~20x faster

#### **Common EXPLAIN Plan Patterns:**

**Pattern 1: Sequential Scan (Table Scan)**
```
Seq Scan on orders  (cost=0..100000 rows=1000000)
  Filter: customer_id = 123
```
**Interpretation:** Reading entire table
**Fix:** Add index if filtering on high-selectivity column

**Pattern 2: Index Scan**
```
Index Scan using idx_customer_id  (cost=0..8 rows=5)
  Index Cond: customer_id = 123
```
**Interpretation:** Using index efficiently
**Good:** Fast, targeted access

**Pattern 3: Nested Loop (⚠️ Watch Out)**
```
Nested Loop  (cost=0..50000000 rows=1000000)
  -> Seq Scan on orders  (rows=1000000)
  -> Index Scan on customers  (rows=1 per loop)
```
**Problem:** Loops 1M times = slow
**When OK:** Small outer table (<1000 rows)

**Pattern 4: Hash Join (Usually Good)**
```
Hash Join  (cost=10000..50000 rows=100000)
  Hash Cond: o.customer_id = c.customer_id
  -> Seq Scan on orders  (rows=1000000)
  -> Hash (Seq Scan on customers)  (rows=50000)
```
**Interpretation:** Build hash table from smaller table (customers)
**Good for:** Medium-sized joins

#### **Databricks/Spark-Specific Plans:**

```sql
EXPLAIN FORMATTED
SELECT * FROM large_table
WHERE partition_col = '2024-01-01';
```

**Key Elements:**
```
== Physical Plan ==
*(1) Filter (partition_col = 2024-01-01)
+- FileScan parquet [...]  ← File format
   PartitionFilters: [partition_col = 2024-01-01]  ← Partition pruning!
   PushedFilters: []  ← Predicate pushdown
   ReadSchema: struct<...>
```

**Look for:**
- **PartitionFilters:** Partition pruning working?
- **PushedFilters:** Filters pushed to storage?
- **BroadcastExchange:** Small table broadcast?
- **Exchange:** Expensive shuffle operations

#### **Optimization Checklist:**

**From EXPLAIN, check:**
✅ Are indexes being used?
✅ Are filters applied early (before joins)?
✅ Is join order optimal (small table first)?
✅ Are row estimates accurate?
✅ Are there unnecessary sorts?
✅ Are partitions being pruned?
✅ Is there excessive data shuffling (Spark)?

#### **Interview Follow-Up:**

**Q: "What's the difference between 'estimated rows' and 'actual rows'?"**

**A:** 
- **Estimated:** Optimizer's guess (based on statistics)
- **Actual:** Real number after execution (requires EXPLAIN ANALYZE)
- **Mismatch = bad statistics** → Run ANALYZE TABLE

**Q: "When is a table scan better than an index scan?"**

**A:** 
- Returning large % of table (>20-30%)
- Table is very small (fits in memory)
- No suitable index exists
- Index selectivity is poor

In [0]:
%sql
-- Demo: EXPLAIN Plans and Query Optimization

-- Create sample tables
CREATE OR REPLACE TABLE workspace.default.customers_perf (
  customer_id INT,
  name STRING,
  email STRING,
  created_date DATE
);

CREATE OR REPLACE TABLE workspace.default.orders_perf (
  order_id INT,
  customer_id INT,
  order_date DATE,
  amount DECIMAL(10,2),
  status STRING
);

-- Insert sample data
INSERT INTO workspace.default.customers_perf
SELECT 
  id AS customer_id,
  CONCAT('Customer_', id) AS name,
  CONCAT('customer', id, '@email.com') AS email,
  DATE_ADD('2023-01-01', id % 365) AS created_date
FROM RANGE(1000);

INSERT INTO workspace.default.orders_perf
SELECT 
  id AS order_id,
  (id % 1000) + 1 AS customer_id,
  DATE_ADD('2024-01-01', id % 90) AS order_date,
  CAST(50 + (id % 200) AS DECIMAL(10,2)) AS amount,
  CASE WHEN id % 4 = 0 THEN 'completed' ELSE 'pending' END AS status
FROM RANGE(5000);

-- Slow query (no filter before join)
EXPLAIN FORMATTED
SELECT c.name, COUNT(o.order_id) AS order_count
FROM workspace.default.customers_perf c
JOIN workspace.default.orders_perf o 
  ON c.customer_id = o.customer_id
WHERE o.order_date >= '2024-02-01'
GROUP BY c.name;

## 🗂️ Section 2: Indexing Strategies (4 Questions)

Indexes speed up reads but slow down writes. Knowing when and how to index is crucial.

### ❓ Question 2: Indexing Strategy

**Critical Interview Question:**
> "You have a users table with 100M rows. Queries filter by email, created_date, and country. Most queries filter by multiple columns. What indexes would you create and why? What are the trade-offs?"

### ✅ Answer 2: Indexing Strategy and Design

#### **When to Create an Index:**

✅ **DO Index:**
- Columns in WHERE clauses (high selectivity)
- Foreign key columns (JOIN conditions)
- Columns in ORDER BY (if result set is small)
- Columns in GROUP BY (sometimes)

❌ **DON'T Index:**
- Low-cardinality columns (gender: M/F, boolean)
- Columns rarely queried
- Small tables (<1000 rows)
- Columns with frequent updates
- Every column (index maintenance cost)

#### **Index Selectivity:**

```sql
-- High selectivity = Good for indexing
SELECT COUNT(DISTINCT email) / COUNT(*) AS selectivity
FROM users;
-- Result: 0.99 (99% unique) → Great for indexing

-- Low selectivity = Bad for indexing
SELECT COUNT(DISTINCT country) / COUNT(*) AS selectivity
FROM users;
-- Result: 0.001 (0.1% unique, ~200 countries) → Poor for indexing
```

**Rule of thumb:** Selectivity >5% → Consider indexing

#### **Index Types:**

**1. Single-Column Index**
```sql
CREATE INDEX idx_users_email ON users(email);

-- Good for:
SELECT * FROM users WHERE email = 'alice@example.com';
```

**2. Composite (Multi-Column) Index**
```sql
CREATE INDEX idx_users_country_date ON users(country, created_date);

-- Works for:
WHERE country = 'US' AND created_date >= '2024-01-01'  -- ✅ Uses index
WHERE country = 'US'                                   -- ✅ Uses index (left prefix)
WHERE created_date >= '2024-01-01'                     -- ❌ Doesn't use index!
```

**Column Order Matters!**
- Most selective column first
- Columns in WHERE before ORDER BY
- Follow query patterns

**3. Covering Index**
```sql
CREATE INDEX idx_users_country_date_name 
  ON users(country, created_date) 
  INCLUDE (name, email);

-- Query can be satisfied entirely from index (no table lookup)
SELECT name, email 
FROM users 
WHERE country = 'US' AND created_date >= '2024-01-01';
-- → Index-only scan (very fast!)
```

**4. Unique Index**
```sql
CREATE UNIQUE INDEX idx_users_email_unique ON users(email);
-- Enforces uniqueness + fast lookups
```

**5. Partial Index (Filtered)**
```sql
CREATE INDEX idx_active_users 
  ON users(created_date) 
  WHERE is_active = TRUE;

-- Smaller index, faster for queries on active users only
-- Not supported in all databases
```

#### **Composite Index Strategy:**

**Scenario:** Queries on (country, created_date, status)

**Bad Approach: 3 separate indexes**
```sql
CREATE INDEX idx1 ON users(country);
CREATE INDEX idx2 ON users(created_date);
CREATE INDEX idx3 ON users(status);
-- Database picks ONE index, others wasted
```

**Good Approach: 1 composite index**
```sql
CREATE INDEX idx_users_composite 
  ON users(country, created_date, status);

-- Handles all query patterns:
WHERE country = 'US'                                              -- ✅
WHERE country = 'US' AND created_date >= '2024-01-01'             -- ✅
WHERE country = 'US' AND created_date >= '2024-01-01' AND status = 'active'  -- ✅

-- But NOT:
WHERE created_date >= '2024-01-01'                                -- ❌ (skips first column)
WHERE status = 'active'                                           -- ❌ (skips first two)
```

**Composite Index Rule: Left-to-Right Prefix**

Index on (A, B, C) works for:
- WHERE A
- WHERE A AND B
- WHERE A AND B AND C

But NOT:
- WHERE B
- WHERE C
- WHERE B AND C

#### **Example: Optimizing Multi-Column Queries**

**Query Pattern Analysis:**
```sql
-- Query 1 (80% of traffic)
SELECT * FROM users 
WHERE country = 'US' 
  AND created_date >= '2024-01-01'
ORDER BY created_date DESC
LIMIT 100;

-- Query 2 (15% of traffic)
SELECT * FROM users 
WHERE email = 'alice@example.com';

-- Query 3 (5% of traffic)
SELECT * FROM users 
WHERE country = 'US' 
  AND status = 'active';
```

**Optimal Index Strategy:**
```sql
-- Index 1: Handles Query 1 (80%)
CREATE INDEX idx_country_date 
  ON users(country, created_date DESC);
-- DESC matches ORDER BY

-- Index 2: Handles Query 2 (15%)
CREATE INDEX idx_email ON users(email);
-- High selectivity, unique lookups

-- Index 3: Handles Query 3 (5%) + part of Query 1
CREATE INDEX idx_country_status 
  ON users(country, status);
```

#### **Index Trade-Offs:**

**Benefits:**
✅ Faster SELECT queries
✅ Faster JOINs
✅ Faster ORDER BY (if index matches sort)

**Costs:**
❌ Slower INSERT/UPDATE/DELETE (index maintenance)
❌ Storage overhead (indexes take space)
❌ More indexes = optimizer has more choices (can choose wrong one)

**Example:**
```
Table: 10GB
Indexes: 5 indexes × 2GB each = 10GB
Total storage: 20GB (2x table size!)

INSERT performance:
- No indexes: 1000 rows/sec
- 5 indexes: 200 rows/sec (5x slower)
```

#### **When NOT to Index:**

**1. Small Tables**
```sql
-- Table with 100 rows: table scan is faster than index
-- Index overhead > benefit
```

**2. High Write Frequency**
```sql
-- Streaming data with 10K inserts/sec
-- Index maintenance becomes bottleneck
-- Consider batch loading or fewer indexes
```

**3. Low Selectivity Columns**
```sql
-- Boolean column (true/false)
CREATE INDEX idx_is_active ON users(is_active);  -- ❌ Bad
-- Returns ~50% of rows → table scan is faster
```

**4. Columns with Expressions**
```sql
-- Index not used
WHERE UPPER(email) = 'ALICE@EXAMPLE.COM'

-- Fix: Function-based index (if supported)
CREATE INDEX idx_email_upper ON users(UPPER(email));
-- Or store computed column
```

#### **Index Maintenance:**

**Update Statistics:**
```sql
-- After bulk loads or significant changes
ANALYZE TABLE users COMPUTE STATISTICS;

-- Databricks: Automatic for Delta tables
OPTIMIZE users;
```

**Rebuild Fragmented Indexes:**
```sql
-- Traditional databases
REINDEX TABLE users;
ALTER INDEX idx_users_email REBUILD;

-- Databricks: Use OPTIMIZE
OPTIMIZE users ZORDER BY (country, created_date);
```

#### **Interview Follow-Up:**

**Q: "How do you decide which columns to index?"**

**A:**
1. Profile queries (find slow ones)
2. Check WHERE, JOIN, ORDER BY columns
3. Calculate selectivity
4. Consider query frequency (80/20 rule)
5. Test impact (measure before/after)

**Q: "Can you have too many indexes?"**

**A:** Yes!
- Each index slows writes
- Storage overhead
- Optimizer confusion (too many choices)
- **Rule of thumb:** 5-7 indexes per table max
- Focus on high-impact queries (80/20 rule)

## 🗃️ Section 3: Partitioning & Data Layout (3 Questions)

Proper partitioning and clustering dramatically improve query performance by reducing data scanned.

### ❓ Question 3: Table Partitioning Design

**Data Engineering Interview Question:**
> "You have an events table with 10 billion rows growing by 100M daily. Queries typically filter by date and user_id. Design a partitioning and clustering strategy. What are the trade-offs?"

### ✅ Answer 3: Partitioning and Clustering Strategies

#### **What is Partitioning?**

Partitioning divides a large table into smaller, manageable pieces based on column values.

**Benefits:**
✅ **Partition Pruning** - Skip irrelevant partitions (faster queries)
✅ **Parallel Processing** - Process partitions in parallel
✅ **Easier Maintenance** - Drop old partitions instead of DELETE
✅ **Better Compression** - Similar data compresses better

**Costs:**
❌ **Small Files Problem** - Too many partitions = slow metadata operations
❌ **Partition Skew** - Uneven partition sizes hurt performance
❌ **Write Overhead** - Partitioning logic on every write

#### **Partitioning Strategies:**

**1. Time-Based Partitioning (Most Common)**

```sql
-- Partition by date
CREATE TABLE events (
  event_id BIGINT,
  user_id BIGINT,
  event_type STRING,
  created_at TIMESTAMP,
  event_date DATE
)
PARTITIONED BY (event_date);

-- Query with partition filter
SELECT COUNT(*)
FROM events
WHERE event_date = '2024-01-15';  -- Reads only 1 partition!
```

**Granularity Choice:**
```
Daily:   365 partitions/year   (Good for most use cases)
Hourly:  8,760 partitions/year (Fine-grained, small files risk)
Monthly: 12 partitions/year    (Coarse, large partitions)
```

**2. Category-Based Partitioning**

```sql
-- Partition by country
CREATE TABLE users (
  user_id BIGINT,
  name STRING,
  country STRING
)
PARTITIONED BY (country);

-- Good when queries always filter by country
SELECT * FROM users WHERE country = 'US';
```

**⚠️ Watch Out:**
- High-cardinality columns (user_id) = too many partitions
- Low-cardinality with skew (90% in one partition) = no benefit

**3. Multi-Level Partitioning**

```sql
-- Partition by year, then month
CREATE TABLE events (
  event_id BIGINT,
  user_id BIGINT,
  event_data STRING
)
PARTITIONED BY (year INT, month INT);

-- Partition structure:
events/year=2024/month=1/
events/year=2024/month=2/
events/year=2024/month=3/
```

**4. Hash Partitioning (Distributed Systems)**

```sql
-- Distribute data evenly across partitions
CREATE TABLE users (
  user_id BIGINT,
  name STRING
)
PARTITIONED BY (HASH(user_id) INTO 100 BUCKETS);

-- Good for: Even distribution, parallel processing
-- Bad for: Range queries (can't prune partitions by user_id range)
```

#### **Databricks: Partitioning + Z-Ordering**

```sql
-- Partition by date, Z-Order by user_id
CREATE TABLE events (
  event_id BIGINT,
  user_id BIGINT,
  event_type STRING,
  event_date DATE
)
USING DELTA
PARTITIONED BY (event_date);

-- Optimize with Z-Ordering
OPTIMIZE events ZORDER BY (user_id);

-- Now queries filter efficiently on BOTH columns:
SELECT *
FROM events
WHERE event_date = '2024-01-15'      -- Partition pruning
  AND user_id = 12345;               -- Z-Order skip
```

**Z-Ordering:**
- Colocation technique (similar values stored together)
- Works like multi-dimensional clustering
- Choose columns frequently filtered together

#### **Liquid Clustering (Databricks)**

```sql
-- Automatic clustering (better than manual Z-Order)
CREATE TABLE events (
  event_id BIGINT,
  user_id BIGINT,
  event_type STRING,
  event_date DATE
)
USING DELTA
CLUSTER BY (event_date, user_id);

-- Databricks auto-optimizes layout
-- No need for manual OPTIMIZE ZORDER BY
```

**Benefits over Z-Ordering:**
- Automatic re-clustering on writes
- Better for high-cardinality columns
- More flexible (can change clustering columns)

#### **Partition Sizing Guidelines:**

**Optimal Partition Size:**
```
Partition Size:  1GB - 10GB per partition
File Size:       128MB - 1GB per file
File Count:      < 10,000 files per partition
```

**Too Small:**
```
Problem: 1M partitions with 10KB each
- Metadata overhead (listing partitions is slow)
- Slow query planning
- Small file problem

Fix: Coarser partitioning (daily → monthly)
```

**Too Large:**
```
Problem: 10 partitions with 1TB each
- No partition pruning benefit
- Large data scans
- Memory pressure

Fix: Finer partitioning (monthly → daily)
```

#### **Partition Pruning Example:**

**Without Partition Pruning:**
```sql
-- ❌ Bad: Function prevents partition pruning
SELECT *
FROM events
WHERE YEAR(event_date) = 2024 AND MONTH(event_date) = 1;
-- Scans ALL partitions!
```

**With Partition Pruning:**
```sql
-- ✅ Good: Direct partition column filter
SELECT *
FROM events
WHERE event_date >= '2024-01-01' 
  AND event_date < '2024-02-01';
-- Scans only January 2024 partition!
```

#### **Partitioning Decision Tree:**

```
Q: What column do queries filter on most?
A: Date/Time → Time-based partitioning
A: Category (country, region) → Category partitioning
A: Multiple columns → Multi-level or clustering

Q: How many distinct values?
A: < 100 → Safe to partition
A: 100-1000 → Consider multi-level or clustering
A: > 1000 → Don't partition (use clustering/indexes)

Q: Is data distribution even?
A: Yes → Proceed
A: No (90% in one partition) → Reconsider or use hash partitioning
```

#### **Common Partitioning Patterns:**

**Pattern 1: Event/Log Data**
```sql
PARTITIONED BY (event_date DATE)
CLUSTER BY (user_id, event_type)
-- Time-series data with user filtering
```

**Pattern 2: E-Commerce**
```sql
PARTITIONED BY (order_year, order_month)
CLUSTER BY (customer_id, product_id)
-- Orders by time, filtered by customer/product
```

**Pattern 3: Multi-Tenant SaaS**
```sql
PARTITIONED BY (tenant_id)
-- Isolate tenants, parallel processing
```

**Pattern 4: Geo-Distributed**
```sql
PARTITIONED BY (region, date)
-- Regional data with time filter
```

#### **Partition Maintenance:**

**Drop Old Partitions:**
```sql
-- Fast: Drop partition (instant metadata operation)
ALTER TABLE events DROP IF EXISTS PARTITION (event_date < '2023-01-01');

-- Slow: DELETE (scans and rewrites data)
DELETE FROM events WHERE event_date < '2023-01-01';
```

**Compact Small Files:**
```sql
-- Databricks: Compact files in partition
OPTIMIZE events 
WHERE event_date = '2024-01-15';

-- Result: Many small files → Fewer large files
```

#### **Interview Follow-Up:**

**Q: "When should you NOT partition a table?"**

**A:**
- Small tables (<10GB)
- No common filter column in queries
- High cardinality column (user_id with 1B users)
- Even scans across entire table (no filter benefit)

**Q: "What's the 'small files problem'?"**

**A:** 
- Too many small files (< 10MB each)
- Metadata overhead (listing files is slow)
- Inefficient I/O (many open/close operations)
- **Fix:** OPTIMIZE to compact files

## 🔄 Section 4: Query Rewriting & Optimization (3 Questions)

Rewriting queries can dramatically improve performance without changing table structure.

### ❓ Question 4: Optimizing Slow Queries

**Practical Interview Question:**
> "This query is taking 2 minutes on a 100M row table. How would you optimize it?"
> 
> ```sql
> SELECT customer_id, 
>        (SELECT COUNT(*) FROM orders o WHERE o.customer_id = c.customer_id) AS order_count
> FROM customers c
> WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id);
> ```

### ✅ Answer 4: Query Rewriting and Optimization Techniques

#### **Problem Analysis:**

**Original Query Issues:**
```sql
SELECT customer_id, 
       (SELECT COUNT(*) FROM orders o WHERE o.customer_id = c.customer_id) AS order_count
FROM customers c
WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id);
```

**Problems:**
1. **Correlated subquery** in SELECT (runs once per customer)
2. **EXISTS subquery** in WHERE (runs once per customer)
3. **Scans orders table twice** for each customer
4. **N+1 query problem** at scale

#### **Optimized Solution:**

```sql
-- Single pass with JOIN and aggregation
SELECT 
  c.customer_id,
  COUNT(o.order_id) AS order_count
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id;
```

**Improvements:**
✅ **Single table scan** of orders (vs N scans)
✅ **No correlated subqueries**
✅ **Hash aggregation** instead of N COUNTs
✅ **~100-1000x faster** at scale

#### **Query Rewriting Patterns:**

**Pattern 1: Replace Correlated Subquery with JOIN**

**❌ Slow (Correlated):**
```sql
SELECT 
  p.product_id,
  p.product_name,
  (SELECT AVG(rating) FROM reviews r WHERE r.product_id = p.product_id) AS avg_rating
FROM products p;
-- Scans reviews table once per product!
```

**✅ Fast (JOIN):**
```sql
SELECT 
  p.product_id,
  p.product_name,
  AVG(r.rating) AS avg_rating
FROM products p
LEFT JOIN reviews r ON p.product_id = r.product_id
GROUP BY p.product_id, p.product_name;
-- Single scan of reviews!
```

**Pattern 2: Replace NOT EXISTS with LEFT JOIN**

**❌ Slow (NOT EXISTS):**
```sql
SELECT *
FROM customers c
WHERE NOT EXISTS (
  SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id
);
```

**✅ Fast (LEFT JOIN + NULL check):**
```sql
SELECT c.*
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.customer_id IS NULL;
-- Often faster, but test both!
```

**Pattern 3: Push Filters Down**

**❌ Slow (Filter after JOIN):**
```sql
SELECT c.name, o.order_total
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_date >= '2024-01-01';  -- Filter AFTER join
-- Joins all orders, then filters
```

**✅ Fast (Filter before JOIN):**
```sql
SELECT c.name, o.order_total
FROM customers c
JOIN (
  SELECT * FROM orders WHERE order_date >= '2024-01-01'
) o ON c.customer_id = o.customer_id;
-- Filters first, joins less data
```

**Pattern 4: Replace DISTINCT with GROUP BY**

**❌ Slow (DISTINCT):**
```sql
SELECT DISTINCT customer_id, order_date
FROM orders;
-- May use expensive sort
```

**✅ Fast (GROUP BY):**
```sql
SELECT customer_id, order_date
FROM orders
GROUP BY customer_id, order_date;
-- Often uses hash aggregation (faster)
```

**Pattern 5: Use UNION ALL Instead of UNION**

**❌ Slow (UNION):**
```sql
SELECT customer_id FROM customers_us
UNION
SELECT customer_id FROM customers_eu;
-- Removes duplicates (expensive sort/hash)
```

**✅ Fast (UNION ALL):**
```sql
SELECT customer_id FROM customers_us
UNION ALL
SELECT customer_id FROM customers_eu;
-- No deduplication if you know there are no duplicates
```

**Pattern 6: Avoid SELECT * in Subqueries**

**❌ Slow:**
```sql
SELECT order_id
FROM (
  SELECT * FROM orders  -- Returns all columns
  WHERE order_date >= '2024-01-01'
) subq;
```

**✅ Fast:**
```sql
SELECT order_id
FROM (
  SELECT order_id, order_date FROM orders  -- Only needed columns
  WHERE order_date >= '2024-01-01'
) subq;
```

**Pattern 7: Early Aggregation**

**❌ Slow (Aggregate after JOIN):**
```sql
SELECT 
  c.customer_id,
  SUM(oi.quantity) AS total_quantity
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY c.customer_id;
-- Joins THEN aggregates (processes all item rows)
```

**✅ Fast (Aggregate before JOIN):**
```sql
SELECT 
  c.customer_id,
  agg.total_quantity
FROM customers c
JOIN (
  SELECT 
    o.customer_id,
    SUM(oi.quantity) AS total_quantity
  FROM orders o
  JOIN order_items oi ON o.order_id = oi.order_id
  GROUP BY o.customer_id
) agg ON c.customer_id = agg.customer_id;
-- Aggregates first (reduces rows before final join)
```

#### **Function Optimization:**

**Pattern 8: Avoid Functions on Indexed Columns**

**❌ Slow (Can't use index):**
```sql
WHERE YEAR(order_date) = 2024
WHERE UPPER(email) = 'ALICE@EXAMPLE.COM'
WHERE price * 1.1 > 100
```

**✅ Fast (Uses index):**
```sql
WHERE order_date >= '2024-01-01' AND order_date < '2025-01-01'
WHERE email = 'alice@example.com'  -- Store lowercase
WHERE price > 100 / 1.1
```

**Pattern 9: Simplify CASE Expressions**

**❌ Slow:**
```sql
CASE 
  WHEN age < 18 THEN 'Minor'
  WHEN age >= 18 AND age < 65 THEN 'Adult'
  WHEN age >= 65 THEN 'Senior'
END
```

**✅ Fast:**
```sql
CASE 
  WHEN age < 18 THEN 'Minor'
  WHEN age < 65 THEN 'Adult'  -- Simplified condition
  ELSE 'Senior'
END
```

#### **Data Type Optimization:**

**Pattern 10: Use Appropriate Data Types**

```sql
-- ❌ Bad
CREATE TABLE events (
  event_id VARCHAR(100),        -- Storing INT as string
  created_at VARCHAR(50),       -- Storing TIMESTAMP as string
  is_active VARCHAR(10)         -- Storing BOOLEAN as string
);
-- String comparisons/sorts are slower

-- ✅ Good
CREATE TABLE events (
  event_id BIGINT,              -- Native INT
  created_at TIMESTAMP,         -- Native TIMESTAMP
  is_active BOOLEAN             -- Native BOOLEAN
);
-- Faster comparisons, less storage
```

#### **Batch Operations:**

**Pattern 11: Batch INSERTs**

**❌ Slow (One at a time):**
```python
for row in data:
    cursor.execute("INSERT INTO table VALUES (?)", row)
# 10,000 rows = 10,000 round trips
```

**✅ Fast (Batch):**
```python
cursor.executemany("INSERT INTO table VALUES (?)", data)
# 10,000 rows = 1 round trip
```

#### **Materialized Views:**

**When to Use:**
```sql
-- Expensive query run frequently
CREATE MATERIALIZED VIEW customer_summary AS
SELECT 
  customer_id,
  COUNT(order_id) AS order_count,
  SUM(order_total) AS lifetime_value
FROM orders
GROUP BY customer_id;

-- Query becomes instant
SELECT * FROM customer_summary WHERE customer_id = 123;
```

**Trade-offs:**
✅ Fast reads
❌ Stale data (refresh lag)
❌ Storage overhead
❌ Maintenance cost

#### **Query Hints (Use Sparingly):**

```sql
-- Force specific join method (when optimizer gets it wrong)
SELECT /*+ BROADCAST(small_table) */ 
  l.*, s.*
FROM large_table l
JOIN small_table s ON l.key = s.key;

-- Force index usage
SELECT /*+ INDEX(orders idx_customer_date) */
  * FROM orders WHERE customer_id = 123;
```

**⚠️ Warning:** Hints bypass optimizer. Use only when:
- You know more than optimizer
- Statistics are stale
- Temporary workaround

#### **Interview Follow-Up:**

**Q: "How do you know if your optimization worked?"**

**A:** Measure!
1. Run EXPLAIN before/after
2. Time query execution
3. Check rows scanned
4. Monitor resource usage (CPU, memory, I/O)
5. Test at production scale

**Q: "When should you stop optimizing?"**

**A:**
- Query meets SLA (e.g., < 1 second)
- Cost of optimization > benefit
- Optimization makes code unmaintainable
- Diminishing returns (95% faster vs 96% faster)

## ⚠️ Section 5: Performance Anti-Patterns (2 Questions)

Common mistakes that kill query performance. Knowing what NOT to do is as important as knowing what to do.

### ❓ Question 5: Identify Performance Anti-Patterns

**Debugging Interview Question:**
> "Review this query and identify all performance issues. How would you fix each one?"
> 
> ```sql
> SELECT *
> FROM orders o, customers c, products p, order_items oi
> WHERE o.customer_id = c.customer_id
>   AND oi.order_id = o.order_id
>   AND oi.product_id = p.product_id
>   AND YEAR(o.order_date) = 2024
>   AND o.order_id IN (SELECT order_id FROM order_items WHERE quantity > 10)
> ORDER BY o.order_date
> LIMIT 100;
> ```

### ✅ Answer 5: Common Performance Anti-Patterns and Fixes

#### **Anti-Pattern Analysis:**

**Issues in the Query:**

1. **❌ SELECT *** - Returns unnecessary columns
2. **❌ Implicit JOINs** (comma syntax) - Hard to read, error-prone
3. **❌ Function on indexed column** - YEAR(order_date) prevents index use
4. **❌ Correlated subquery** - IN with subquery scans order_items multiple times
5. **❌ No index hints for ORDER BY** - May sort large result set
6. **❌ LIMIT without ORDER BY tiebreaker** - Non-deterministic results

#### **Fixed Query:**

```sql
SELECT 
  o.order_id,
  o.order_date,
  c.customer_name,
  p.product_name,
  oi.quantity
FROM orders o
INNER JOIN order_items oi 
  ON o.order_id = oi.order_id
  AND oi.quantity > 10  -- Filter early
INNER JOIN customers c 
  ON o.customer_id = c.customer_id
INNER JOIN products p 
  ON oi.product_id = p.product_id
WHERE o.order_date >= '2024-01-01' 
  AND o.order_date < '2025-01-01'  -- Index-friendly
ORDER BY o.order_date DESC, o.order_id DESC  -- Deterministic
LIMIT 100;
```

#### **Top 10 Performance Anti-Patterns:**

**1. SELECT * Anti-Pattern**

**❌ Problem:**
```sql
SELECT * FROM large_table;
-- Returns all columns (including BLOBs, large TEXT)
-- Wastes bandwidth, memory, I/O
```

**✅ Fix:**
```sql
SELECT id, name, email FROM large_table;
-- Only needed columns
```

**Impact:** 10-100x data transfer reduction

---

**2. N+1 Query Problem**

**❌ Problem:**
```python
# Fetch customers
customers = execute("SELECT * FROM customers")
for customer in customers:
    # N queries (one per customer!)
    orders = execute(f"SELECT * FROM orders WHERE customer_id = {customer.id}")
# 1 query + N queries = N+1
```

**✅ Fix:**
```python
# Single query with JOIN
result = execute("""
    SELECT c.*, o.*
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
""")
# 1 query total
```

---

**3. Unindexed Foreign Keys**

**❌ Problem:**
```sql
CREATE TABLE orders (
  order_id INT PRIMARY KEY,
  customer_id INT  -- No index!
);

-- Slow JOIN (table scan)
SELECT * FROM customers c
JOIN orders o ON c.customer_id = o.customer_id;
```

**✅ Fix:**
```sql
CREATE INDEX idx_orders_customer ON orders(customer_id);
```

---

**4. Using OR Instead of IN**

**❌ Problem:**
```sql
WHERE customer_id = 1 
   OR customer_id = 2 
   OR customer_id = 3
   OR customer_id = 4
   ...  -- Can't use index efficiently
```

**✅ Fix:**
```sql
WHERE customer_id IN (1, 2, 3, 4, ...)
-- Or use temporary table for large lists
```

---

**5. Wildcard at Start of LIKE**

**❌ Problem:**
```sql
WHERE email LIKE '%@example.com'
-- Can't use index (needs full table scan)
```

**✅ Fix:**
```sql
-- If possible, rewrite
WHERE email LIKE 'user%@example.com'
-- Or use full-text search index
```

---

**6. Not Using LIMIT**

**❌ Problem:**
```sql
SELECT * FROM orders
ORDER BY order_date DESC;
-- Returns ALL rows (millions!), then client takes first 100
```

**✅ Fix:**
```sql
SELECT * FROM orders
ORDER BY order_date DESC
LIMIT 100;
-- Database stops after 100 rows
```

---

**7. Cartesian Product (Missing JOIN Condition)**

**❌ Problem:**
```sql
SELECT * FROM orders, customers;
-- No JOIN condition = every order × every customer!
-- 1M orders × 100K customers = 100B rows!
```

**✅ Fix:**
```sql
SELECT * FROM orders o
JOIN customers c ON o.customer_id = c.customer_id;
```

---

**8. Using COUNT(*) When You Just Need EXISTS**

**❌ Problem:**
```sql
-- Check if orders exist for customer
IF (SELECT COUNT(*) FROM orders WHERE customer_id = 123) > 0 THEN
-- Counts ALL orders (expensive)
```

**✅ Fix:**
```sql
IF EXISTS (SELECT 1 FROM orders WHERE customer_id = 123) THEN
-- Stops at first row (fast)
```

---

**9. Polling Instead of Triggers/Streams**

**❌ Problem:**
```sql
-- Poll every second for new records
SELECT * FROM events 
WHERE created_at > last_processed_time;
-- Constant load, wasted queries when no new data
```

**✅ Fix:**
- Use database triggers
- Use change data capture (CDC)
- Use message queue (Kafka)

---

**10. Not Using Prepared Statements**

**❌ Problem:**
```python
# String concatenation (SQL injection risk + no plan caching)
query = f"SELECT * FROM users WHERE id = {user_id}"
execute(query)
```

**✅ Fix:**
```python
# Prepared statement (safe + cached execution plan)
query = "SELECT * FROM users WHERE id = ?"
execute(query, [user_id])
```

#### **Data Skew Anti-Patterns:**

**Problem: Hot Partition**
```sql
-- 90% of data in one partition
PARTITIONED BY (country)
-- WHERE country = 'US' → hits single large partition
```

**Fix:**
```sql
-- Add salt for even distribution
PARTITIONED BY (country, HASH(customer_id) % 10)
-- Splits large partitions into sub-partitions
```

#### **Interview Follow-Up:**

**Q: "What's the first thing you check when a query is slow?"**

**A:** Run EXPLAIN and check:
1. Table scans (should be index seeks)
2. Row estimates (should match actual)
3. Join method (nested loop on large table = bad)
4. Filter position (should be early)

**Q: "How do you prevent performance issues in production?"**

**A:**
1. **Code review** - Check for anti-patterns
2. **Test at scale** - 1000 rows ≠ 1B rows
3. **Monitor slow queries** - Log queries >1 second
4. **Query complexity limits** - Max joins, max subqueries
5. **Resource quotas** - Timeout after N seconds

## 🎓 Congratulations - You've Mastered SQL Performance!

### 📊 What You've Learned:

✅ **EXPLAIN Plans** - Reading and interpreting execution plans
✅ **Indexing Strategies** - When to index, composite indexes, selectivity
✅ **Partitioning** - Time-based, Z-ordering, liquid clustering
✅ **Query Rewriting** - Transforming slow queries to fast ones
✅ **Anti-Patterns** - Common mistakes and how to avoid them
✅ **JOIN Optimization** - Join order, methods, and strategies
✅ **Performance Metrics** - What to measure and why

---

### 🚀 Key Takeaways:

**Performance Hierarchy (Fastest to Slowest):**
```
1. Index seek              (microseconds)
2. Index scan              (milliseconds)
3. Table scan (small)      (milliseconds)
4. Hash join               (seconds)
5. Table scan (large)      (seconds to minutes)
6. Nested loop (large)     (minutes to hours)
7. Cartesian product       (hours to never)
```

**Indexing Rules:**
- High selectivity (>5% unique) = Good candidate
- Foreign keys = Always index
- WHERE/JOIN columns = Index
- Low cardinality (<100 distinct) = Don't index
- Too many indexes (>7) = Diminishing returns

**Partitioning Guidelines:**
- Partition size: 1-10GB per partition
- File size: 128MB-1GB per file
- Time-based for logs/events
- Category-based for multi-tenant

**Query Optimization Checklist:**
✅ SELECT only needed columns
✅ Filter early (before JOINs)
✅ Use indexes (avoid functions on indexed columns)
✅ Replace correlated subqueries with JOINs
✅ Use LIMIT for pagination
✅ Choose appropriate data types
✅ Batch operations when possible

**Anti-Pattern Red Flags:**
❌ SELECT *
❌ N+1 queries
❌ Missing indexes on foreign keys
❌ LIKE '%pattern'
❌ Functions on indexed columns
❌ Unintentional Cartesian products

---

### 🎯 Interview Day Checklist:

✅ Can you read an EXPLAIN plan?
✅ Do you know when to add indexes?
✅ Can you explain index selectivity?
✅ Do you understand partitioning trade-offs?
✅ Can you rewrite slow queries?
✅ Do you know common anti-patterns?
✅ Can you calculate query complexity?
✅ Do you test at production scale?

---

### 📚 Next Steps:

1. **Practice Profiling** - Run EXPLAIN on every query
2. **Benchmark Everything** - Measure before/after optimization
3. **Study Production Issues** - Learn from real slow queries
4. **Advanced Topics** - Move to Module 5 (Subqueries & Set Operations)
5. **Distributed Systems** - Learn Spark/Databricks-specific optimizations

---

### 💡 Final Interview Tips:

**Always start with questions:**
- "How many rows in the table?"
- "What's the data growth rate?"
- "What queries run most frequently?"
- "What's the current performance?"
- "What's the performance SLA?"

**Walk through your thinking:**
- "I'd start by running EXPLAIN..."
- "Looking at the execution plan, I see..."
- "The bottleneck appears to be..."
- "I'd test this optimization by..."

**Mention trade-offs:**
- "This index speeds reads but slows writes"
- "Partitioning helps query performance but complicates maintenance"
- "Denormalization improves speed but increases storage"

**Show production awareness:**
- "This works on sample data, but at scale..."
- "We'd need to monitor query execution time..."
- "I'd set up alerts for slow queries..."

---

**You're now ready for performance optimization interviews!** 🎉

Good luck! 🚀

*Pro tip: The best optimization is often the simplest. Don't over-engineer. Profile first, optimize second, and always measure the impact!*